# Quantum-Guided Cluster Algorithm for Max-Cut

Use QAOA or PCE two-point correlations to guide classical cluster moves for Max-Cut.

**Reference:** [arXiv:2508.10656](https://arxiv.org/abs/2508.10656) — Eder et al., Amazon Quantum Solutions Lab

In [ ]:
import time

from main import run_benchmark

## The Algorithm

Classical methods like simulated annealing make small, local moves that easily get trapped in rugged energy landscapes. The QGCA solves this by using **quantum-derived correlations** to guide cluster formation.

1. **Quantum Phase:** QAOA extracts pairwise correlations ⟨Z_i Z_j⟩ — how spins tend to align in good solutions
2. **Classical Phase:** Cluster Monte Carlo uses these correlations to build meaningful spin clusters, enabling large, coordinated moves through the search space

### Key Divi Features

| Feature | Role |
|---------|------|
| **QAOA** with `.MAXCUT` | Extracts two-point correlations |
| **QWC Observable Grouping** | Up to **60% circuit reduction** |
| **QDrift Trotterization** | Randomized Trotter for shallower circuits at high depth |

---

## Local example (10 nodes)

Small graph to prove the algorithm works. Runs in seconds on any laptop.

In [ ]:
t0 = time.time()
results_local = run_benchmark(
    n_nodes=10,
    degree=6,
    qaoa_depths=[1, 2],
    n_iterations_factor=200,
    n_repetitions=10,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=5_000,
    output_dir="plots",
)
phase1_time = time.time() - t0
print(f"\nLocal run completed in {phase1_time:.1f}s")

---

## Paper benchmark configuration (28 nodes)

The paper's primary benchmark: **28-node, 10-regular graphs** at depths p=1, 2, 3, 5. Each QAOA run is dispatched to QoroService, and the deeper runs use **QDrift** trotterization — randomized Trotter sampling that produces shallower circuits at high p without sacrificing the exact dynamics in expectation.

**Requirements:**
Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```
For a local run, use the smaller configuration in the first example.

In [ ]:
print("☁️  Routing 28-qubit QAOA circuits to Qoro Maestro...")

t0 = time.time()
results_cloud = run_benchmark(
    n_nodes=28,
    degree=10,
    qaoa_depths=[1, 2, 3, 5],
    use_qdrift=True,  # randomized Trotter — shallower circuits at p=5
    n_iterations_factor=500,
    n_repetitions=30,
    lambda_scale=4,
    seed=42,
    use_cloud=True,
    shots=10_000,
    output_dir="plots",
)
phase2_time = time.time() - t0

print(f"\n⚡ Local  (Phase 1): {phase1_time:.1f}s for 10 nodes")
print(f"⚡ Cloud  (Phase 2): {phase2_time:.1f}s for 28 nodes")

## Multi-seed check

One graph can be atypical. Repeat a small local configuration across several graph seeds and compare the range of QAOA-guided approximation ratios.

In [ ]:
summary = run_multiseed_benchmark(
    seeds=[1, 2, 3],
    n_nodes=8,
    degree=3,
    qaoa_depths=[1],
    n_iterations_factor=10,
    n_repetitions=2,
    lambda_scale=4,
    use_cloud=False,
    shots=1_000,
    output_dir="plots/multiseed",
)
summary

---

## Variation — Swap QAOA for PCE

The cluster algorithm only needs a **correlation matrix**; it doesn't care whether that matrix came from QAOA, [Pauli Correlation Encoding](https://arxiv.org/abs/2401.09421), or anywhere else. Both Divi extractors return the same `CorrelationResult`, so switching is a one-line change.

PCE uses a different encoding of the optimization variables:

- **Dense:** ⌈log₂(N+1)⌉ qubits  (16 vars → 5 qubits)
- **Poly:** ⌈√(2N)⌉ qubits        (16 vars → 6 qubits)

`run_benchmark` accepts `pce_encodings` alongside `qaoa_depths`, putting both kinds of source on the same comparison plots.

In [ ]:
results_pce = run_benchmark(
    n_nodes=16,
    degree=10,
    qaoa_depths=[2],  # one QAOA reference at the transition depth
    pce_encodings=["dense", "poly"],  # plus both PCE encodings on the same plots
    n_iterations_factor=500,
    n_repetitions=20,
    lambda_scale=4,
    seed=42,
    use_cloud=False,
    shots=10_000,
    output_dir="plots_pce",
)

---

## Explore further

Compare QAOA depths, PCE encodings, and cluster-link parameters on a graph size that fits your local environment.